# J1 — Préparer la population des communes d’Île-de-France

Ce notebook :

1. repère le dossier `GeoMarketing_IDF` dans OneDrive ;
2. charge la base de population 2022 et le COG 2025 ;
3. conserve uniquement les communes officielles (`TYPECOM = COM`) ;
4. traite automatiquement les communes fusionnées ;
5. effectue la jointure sur `CODGEO` ;
6. filtre l’Île-de-France (`REG = 11`) ;
7. contrôle notamment Saint-Denis et Pierrefitte-sur-Seine ;
8. exporte le résultat dans `data/processed/insee`.

> Les fichiers placés dans `data/raw` ne sont jamais modifiés. Exécute les cellules dans l’ordre avec **Shift + Entrée**.

## 1. Configuration

Le notebook recherche automatiquement OneDrive. Si cette détection échoue, renseigne seulement `RACINE_PROJET_MANUELLE` dans la cellule suivante, par exemple :

```python
RACINE_PROJET_MANUELLE = r"C:\\Users\\Kassim\\OneDrive\\GeoMarketing_IDF"
```

In [ ]:
from pathlib import Path
import os
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 100)

# Laisse None si ton dossier OneDrive est détecté automatiquement.
# Sinon, remplace None par le chemin complet indiqué dans la cellule précédente.
RACINE_PROJET_MANUELLE = None

if RACINE_PROJET_MANUELLE:
    RACINE_PROJET = Path(RACINE_PROJET_MANUELLE)
else:
    candidats_onedrive = [
        os.getenv("OneDrive"),
        os.getenv("OneDriveConsumer"),
        os.getenv("OneDriveCommercial"),
        str(Path.home() / "OneDrive"),
    ]
    candidats_onedrive = [Path(p) for p in candidats_onedrive if p]
    racines_possibles = [p / "GeoMarketing_IDF" for p in candidats_onedrive]
    RACINE_PROJET = next((p for p in racines_possibles if p.exists()), None)

    if RACINE_PROJET is None:
        chemins_testes = "\n".join(f"- {p}" for p in racines_possibles)
        raise FileNotFoundError(
            "Le dossier GeoMarketing_IDF est introuvable. "
            "Renseigne RACINE_PROJET_MANUELLE.\n"
            f"Chemins testés :\n{chemins_testes}"
        )

DOSSIER_DATA = RACINE_PROJET / "data"
DOSSIER_RAW_INSEE = DOSSIER_DATA / "raw" / "insee"
DOSSIER_SORTIE = DOSSIER_DATA / "processed" / "insee"

print(f"Racine du projet : {RACINE_PROJET}")
print(f"Données brutes Insee : {DOSSIER_RAW_INSEE}")
print(f"Dossier de sortie : {DOSSIER_SORTIE}")

## 2. Repérage automatique des deux CSV

Le ZIP de chaque source doit être extrait. Le notebook cherche :

- le CSV `base-cc-evol-struct-pop-2022...` hors fichier de métadonnées ;
- le CSV `v_commune_2025.csv`.

In [ ]:
def trouver_csv(dossier, fragment_obligatoire, fragments_exclus=()):
    if not dossier.exists():
        raise FileNotFoundError(f"Dossier introuvable : {dossier}")

    fragment_obligatoire = fragment_obligatoire.lower()
    fragments_exclus = tuple(x.lower() for x in fragments_exclus)

    fichiers = [
        p for p in dossier.rglob("*")
        if p.is_file()
        and p.suffix.lower() == ".csv"
        and fragment_obligatoire in p.name.lower()
        and not any(x in p.name.lower() for x in fragments_exclus)
    ]

    if not fichiers:
        raise FileNotFoundError(
            f"Aucun CSV contenant '{fragment_obligatoire}' trouvé dans {dossier}. "
            "Vérifie que le ZIP a bien été extrait."
        )

    if len(fichiers) > 1:
        liste = "\n".join(f"- {p}" for p in fichiers)
        raise RuntimeError(
            f"Plusieurs fichiers correspondent à '{fragment_obligatoire}'. "
            f"Conserve une seule copie :\n{liste}"
        )

    return fichiers[0]

FICHIER_POPULATION = trouver_csv(
    DOSSIER_RAW_INSEE,
    "base-cc-evol-struct-pop-2022",
    fragments_exclus=("meta",),
)

FICHIER_COG = trouver_csv(
    DOSSIER_RAW_INSEE,
    "v_commune_2025",
)

print(f"Population : {FICHIER_POPULATION}")
print(f"COG 2025 : {FICHIER_COG}")

## 3. Chargement des données

La fonction suivante détecte automatiquement si le séparateur est une virgule, un point-virgule ou une tabulation.

In [ ]:
def detecter_separateur(fichier, encodage):
    with fichier.open("r", encoding=encodage) as f:
        premiere_ligne = f.readline()

    separateurs = [";", ",", "\t"]
    comptes = {sep: premiere_ligne.count(sep) for sep in separateurs}
    separateur = max(comptes, key=comptes.get)

    if comptes[separateur] == 0:
        raise ValueError(f"Séparateur impossible à détecter dans {fichier}")

    return separateur


def lire_csv_insee(fichier, dtype=None):
    derniere_erreur = None

    for encodage in ("utf-8-sig", "utf-8", "latin-1"):
        try:
            separateur = detecter_separateur(fichier, encodage)
            tableau = pd.read_csv(
                fichier,
                sep=separateur,
                encoding=encodage,
                dtype=dtype,
                low_memory=False,
            )
            print(
                f"{fichier.name} : séparateur={repr(separateur)}, "
                f"encodage={encodage}, lignes={len(tableau):,}, "
                f"colonnes={len(tableau.columns)}"
            )
            return tableau
        except (UnicodeDecodeError, pd.errors.ParserError) as erreur:
            derniere_erreur = erreur

    raise RuntimeError(f"Impossible de lire {fichier}") from derniere_erreur


population = lire_csv_insee(
    FICHIER_POPULATION,
    dtype={"CODGEO": "string"},
)

cog = lire_csv_insee(
    FICHIER_COG,
    dtype={
        "TYPECOM": "string",
        "COM": "string",
        "REG": "string",
        "DEP": "string",
        "COMPARENT": "string",
    },
)

## 4. Vérification et normalisation des codes

Les codes géographiques sont conservés comme du texte afin de ne jamais perdre les zéros initiaux.

In [ ]:
colonnes_population_requises = {"CODGEO", "P22_POP"}
colonnes_cog_requises = {
    "TYPECOM", "COM", "REG", "DEP", "LIBELLE", "COMPARENT"
}

manquantes_population = colonnes_population_requises - set(population.columns)
manquantes_cog = colonnes_cog_requises - set(cog.columns)

if manquantes_population:
    raise ValueError(
        f"Colonnes absentes de la population : {sorted(manquantes_population)}"
    )

if manquantes_cog:
    raise ValueError(f"Colonnes absentes du COG : {sorted(manquantes_cog)}")

population["CODGEO"] = (
    population["CODGEO"].astype("string").str.strip().str.zfill(5)
)

for colonne in ["TYPECOM", "COM", "REG", "DEP", "LIBELLE", "COMPARENT"]:
    cog[colonne] = cog[colonne].astype("string").str.strip()

cog["TYPECOM"] = cog["TYPECOM"].str.upper()
cog["COM"] = cog["COM"].str.zfill(5)
cog["REG"] = cog["REG"].str.zfill(2)
cog["COMPARENT"] = cog["COMPARENT"].replace("", pd.NA).str.zfill(5)

print("Types de zones présents dans le COG :")
display(cog["TYPECOM"].value_counts(dropna=False).rename("nombre"))

## 5. Contrôle du cas Saint-Denis / Pierrefitte-sur-Seine

Cette cellule montre les lignes brutes du COG. Les lignes `COMD` sont des communes déléguées ; leurs informations géographiques viennent de `COMPARENT`. Elles ne doivent pas entrer dans le tableau final des communes officielles.

In [ ]:
controle_saint_denis_brut = (
    cog.loc[
        cog["COM"].isin(["93059", "93066"])
        | cog["COMPARENT"].eq("93066"),
        ["TYPECOM", "COM", "LIBELLE", "COMPARENT", "REG", "DEP"],
    ]
    .sort_values(["TYPECOM", "COM"])
)

display(controle_saint_denis_brut)

## 6. Création automatique du référentiel des communes franciliennes

Le filtre `TYPECOM = COM` règle automatiquement toutes les fusions présentes dans le COG 2025. Aucune liste manuelle de communes fusionnées n’est nécessaire.

In [ ]:
dim_commune_idf = (
    cog.loc[
        cog["TYPECOM"].eq("COM") & cog["REG"].eq("11"),
        ["COM", "LIBELLE", "REG", "DEP"],
    ]
    .rename(columns={"COM": "CODGEO", "LIBELLE": "NOM_COMMUNE"})
    .sort_values("CODGEO")
    .reset_index(drop=True)
)

doublons_dim = dim_commune_idf.loc[
    dim_commune_idf["CODGEO"].duplicated(keep=False)
]

if not doublons_dim.empty:
    display(doublons_dim)
    raise ValueError("Le référentiel contient des CODGEO dupliqués.")

if dim_commune_idf[["CODGEO", "NOM_COMMUNE", "REG", "DEP"]].isna().any().any():
    lignes_incompletes = dim_commune_idf.loc[
        dim_commune_idf[["CODGEO", "NOM_COMMUNE", "REG", "DEP"]]
        .isna()
        .any(axis=1)
    ]
    display(lignes_incompletes)
    raise ValueError("Certaines communes officielles ont des informations manquantes.")

print(f"Communes officielles franciliennes : {len(dim_commune_idf):,}")
display(dim_commune_idf.loc[dim_commune_idf["CODGEO"].eq("93066")])

## 7. Jointure avec la population

Le référentiel communal est placé à gauche de la jointure. Ainsi, chaque commune officielle francilienne doit apparaître exactement une fois.

In [ ]:
doublons_population = population.loc[
    population["CODGEO"].duplicated(keep=False),
    ["CODGEO"],
]

if not doublons_population.empty:
    display(doublons_population.drop_duplicates().head(100))
    raise ValueError("La base de population contient des CODGEO dupliqués.")

# Le COG est la source de référence pour le nom, le département et la région.
population_sans_geo = population.drop(
    columns=["REG", "DEP", "LIBGEO"],
    errors="ignore",
)

population_idf = dim_commune_idf.merge(
    population_sans_geo,
    on="CODGEO",
    how="left",
    validate="one_to_one",
    indicator=True,
)

print(f"Lignes après jointure : {len(population_idf):,}")
display(population_idf.head())

## 8. Contrôles de qualité

L’export est interrompu si une commune officielle ne trouve pas sa population, si un code est dupliqué ou si Pierrefitte est encore comptée comme commune indépendante.

In [ ]:
non_appariees = population_idf.loc[
    population_idf["_merge"].ne("both"),
    ["CODGEO", "NOM_COMMUNE", "DEP", "REG", "_merge"],
]

if not non_appariees.empty:
    display(non_appariees)
    raise ValueError(
        "Certaines communes franciliennes ne trouvent pas de population. "
        "Vérifie que la population et le COG utilisent bien la géographie 2025."
    )

if population_idf["CODGEO"].duplicated().any():
    raise ValueError("Des CODGEO sont dupliqués après la jointure.")

departements_idf = {"75", "77", "78", "91", "92", "93", "94", "95"}
departements_inattendus = set(population_idf["DEP"].dropna()) - departements_idf

if departements_inattendus:
    raise ValueError(
        f"Départements inattendus dans l’Île-de-France : {departements_inattendus}"
    )

if not population_idf["REG"].eq("11").all():
    raise ValueError("Le résultat contient des lignes hors Île-de-France.")

controle_fusion_final = population_idf.loc[
    population_idf["CODGEO"].isin(["93059", "93066"]),
    ["CODGEO", "NOM_COMMUNE", "DEP", "REG", "P22_POP"],
]

display(controle_fusion_final)

if not population_idf["CODGEO"].eq("93066").any():
    raise ValueError("La commune nouvelle de Saint-Denis (93066) est absente.")

if population_idf["CODGEO"].eq("93059").any():
    raise ValueError(
        "Pierrefitte-sur-Seine (93059) est encore présente comme commune indépendante."
    )

print("Tous les contrôles sont réussis.")

## 9. Préparation et export du résultat

Deux fichiers seront créés dans OneDrive :

- `dim_commune_idf_2025.csv` : le référentiel communal ;
- `population_idf_2022.csv` : le référentiel joint à toutes les variables de population.

In [ ]:
population_idf = population_idf.drop(columns="_merge")

colonnes_prioritaires = [
    "CODGEO", "NOM_COMMUNE", "DEP", "REG", "P22_POP"
]
autres_colonnes = [
    c for c in population_idf.columns if c not in colonnes_prioritaires
]
population_idf = population_idf[colonnes_prioritaires + autres_colonnes]

DOSSIER_SORTIE.mkdir(parents=True, exist_ok=True)

FICHIER_DIM_SORTIE = DOSSIER_SORTIE / "dim_commune_idf_2025.csv"
FICHIER_POP_SORTIE = DOSSIER_SORTIE / "population_idf_2022.csv"

dim_commune_idf.to_csv(
    FICHIER_DIM_SORTIE,
    sep=";",
    index=False,
    encoding="utf-8-sig",
)

population_idf.to_csv(
    FICHIER_POP_SORTIE,
    sep=";",
    index=False,
    encoding="utf-8-sig",
)

print("J1 terminé. Fichiers créés :")
print(f"- {FICHIER_DIM_SORTIE}")
print(f"- {FICHIER_POP_SORTIE}")
print(f"- {len(population_idf):,} communes franciliennes")
print(f"- {len(population_idf.columns):,} colonnes dans la base finale")

## Fin du J1

Lorsque la dernière cellule affiche **J1 terminé**, enregistre le notebook avec `Ctrl + S`, puis place seulement le notebook dans le dossier `notebooks` de ton dépôt GitHub.

Les CSV bruts et les fichiers produits restent dans OneDrive et ne doivent pas être ajoutés à GitHub.